# SSDLite-MobileNetV3-Large for NSFW object detection — with Hyperparameter Tuning

Optuna **RandomSearch** over learning rate, momentum, weight decay, the
classification-loss weight, and batch size, followed by a full-length retrain
with the best configuration. Mirrors the structure of the EfficientDet / NLP
`_hyperparameters` notebooks.

The base pipeline matches `mobilenetssd_torch.ipynb`: torchvision
`ssdlite320_mobilenet_v3_large` (COCO-pretrained) with its head swapped for 4
classes, BatchNorm replaced by GroupNorm, 640×640 input, and the model's
built-in transform handling normalization (so the dataset transform outputs
plain `[0,1]` tensors).

In [ ]:
!pip install roboflow
!pip install torchmetrics
!pip install -q optuna

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
import os

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("renaire-odarve")
project = workspace.project("erotica-detection")
version = project.version(4)  # dataset version on Roboflow
dataset = version.download("voc")

In [ ]:
import os
import torch
from PIL import Image
import xml.etree.ElementTree as ET

CLASSES = ["__background__", "nsfw", "safe", "gore"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)  # = 4

class RoboflowVOCDataset(torch.utils.data.Dataset):
    def __init__(self, root, split="train", transforms=None):
        self.root = root
        self.split = split
        self.transforms = transforms
        self.split_dir = os.path.join(root, split)

        self.images = sorted(
            f for f in os.listdir(self.split_dir) if f.endswith(".jpg")
        )

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.split_dir, img_name)
        xml_path = img_path.replace(".jpg", ".xml")

        image = Image.open(img_path).convert("RGB")

        boxes = []
        labels = []

        if os.path.exists(xml_path):
            tree = ET.parse(xml_path)
            root = tree.getroot()

            for obj in root.findall("object"):
                class_name = obj.find("name").text.strip().lower()

                if class_name not in CLASS_TO_IDX:
                    continue

                label = CLASS_TO_IDX[class_name]

                bndbox = obj.find("bndbox")
                xmin = float(bndbox.find("xmin").text)
                ymin = float(bndbox.find("ymin").text)
                xmax = float(bndbox.find("xmax").text)
                ymax = float(bndbox.find("ymax").text)

                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(label)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

    def __len__(self):
        return len(self.images)

In [ ]:
import torchvision.transforms.functional as F
import torch
import random
from PIL import Image

class SSDTransform:
    # NOTE: We deliberately do NOT normalize here. torchvision's SSD has a
    # built-in GeneralizedRCNNTransform that normalizes every input with
    # ImageNet mean/std (image_mean=[0.485,0.456,0.406], image_std=[0.229,0.224,0.225]).
    # Normalizing here too would normalize each image twice during training,
    # while the exported model only normalizes once -> a train/inference
    # mismatch. We output plain [0,1] tensors and let the model normalize.
    def __init__(self, size=(640, 640), split='train'):
        self.size = size
        self.split = split

    def __call__(self, image, target):
        # image: PIL Image
        # target: dict with 'boxes', 'labels'

        # === Augmentations (Train Only) ===
        if self.split == 'train':
            # 1. Color Jitter
            if random.random() < 0.5:
                image = F.adjust_brightness(image, brightness_factor=random.uniform(0.6, 1.4))
                image = F.adjust_contrast(image, contrast_factor=random.uniform(0.6, 1.4))
                image = F.adjust_saturation(image, saturation_factor=random.uniform(0.6, 1.4))
                image = F.adjust_hue(image, hue_factor=random.uniform(-0.1, 0.1))

            # 2. Random Horizontal Flip
            if random.random() < 0.5:
                image = F.hflip(image)
                w, h = image.size
                boxes = target['boxes']
                if boxes.numel() > 0:
                    # Flip x coordinates: x1_new = w - x2_old
                    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                    target['boxes'] = boxes

            # 3. Scale Jitter / Zoom Out
            # Expands the image canvas (making the objects smaller relative to the new frame)
            if random.random() < 0.5:
                scale = random.uniform(1.0, 1.5)
                w, h = image.size
                new_w = int(w * scale)
                new_h = int(h * scale)

                # Create larger gray canvas
                new_image = Image.new("RGB", (new_w, new_h), (123, 117, 104))
                left = random.randint(0, new_w - w)
                top = random.randint(0, new_h - h)
                new_image.paste(image, (left, top))

                image = new_image

                # Shift boxes
                boxes = target['boxes']
                if boxes.numel() > 0:
                    boxes[:, [0, 2]] += left
                    boxes[:, [1, 3]] += top
                    target['boxes'] = boxes

        # === Resizing (All Splits) ===
        # Resize to a fixed square; normalization is handled by the model's
        # internal transform, so we only convert to a [0,1] tensor here.
        w, h = image.size
        image = F.resize(image, self.size)
        image = F.to_tensor(image)

        # Resize bounding boxes
        boxes = target['boxes']
        if boxes.numel() > 0:
            scale_w = self.size[1] / w
            scale_h = self.size[0] / h

            boxes[:, 0] *= scale_w
            boxes[:, 2] *= scale_w
            boxes[:, 1] *= scale_h
            boxes[:, 3] *= scale_h

            # Clamp boxes to ensure they stay within image bounds
            boxes[:, 0].clamp_(min=0, max=self.size[1])
            boxes[:, 1].clamp_(min=0, max=self.size[0])
            boxes[:, 2].clamp_(min=0, max=self.size[1])
            boxes[:, 3].clamp_(min=0, max=self.size[0])

            target['boxes'] = boxes

        return image, target

# Create separate transforms
train_transform = SSDTransform(split='train')
val_transform = SSDTransform(split='valid')

In [ ]:
import torch
import torch.nn as nn
from functools import partial
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection import SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection.ssdlite import SSDLiteHead


def get_in_channels(module):
    for layer in module.modules():
        if isinstance(layer, nn.Conv2d):
            return layer.in_channels
    raise ValueError(f"Could not find Conv2d in {module}")


def replace_bn_with_gn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            groups = min(32, child.num_features)
            while child.num_features % groups != 0:
                groups -= 1
            setattr(module, name, nn.GroupNorm(groups, child.num_features))
        else:
            replace_bn_with_gn(child)


def build_model(num_classes, input_size=640):
    """Recreate the exact architecture used in mobilenetssd_torch.ipynb.

    Fresh model per Optuna trial so each trial starts from the same
    COCO-pretrained weights (downloaded once, then cached).
    """
    model = ssdlite320_mobilenet_v3_large(
        weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1,
        trainable_backbone_layers=6,
    )
    for p in model.backbone.parameters():
        p.requires_grad = True

    # Allow 640x640 input
    model.transform.min_size = (input_size,)
    model.transform.max_size = input_size

    # Swap the head for our number of classes
    in_channels = [get_in_channels(layer) for layer in model.head.classification_head.module_list]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head = SSDLiteHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        norm_layer=partial(nn.BatchNorm2d, eps=0.001, momentum=0.03),
    )

    # BatchNorm -> GroupNorm: batch-size robust, which matters because HPO
    # samples different batch sizes across trials.
    replace_bn_with_gn(model)
    return model

# HYPERPARAMETER OPTIMIZATION (Optuna RandomSearch)

Each trial trains for a few short `SEARCH_EPOCHS` on a capped number of batches
and is scored by **validation loss** (`classification * cls_weight +
bbox_regression`) — a stable signal in few epochs. Unpromising trials are
pruned early. The best config is then retrained for the full `NUM_EPOCHS` with
cosine LR decay, per-epoch checkpointing (so a Colab disconnect doesn't lose the
run), and best-by-mAP model saving.

In [ ]:
import gc
import time
import random
import numpy as np
from torch.utils.data import DataLoader

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE   = 640
NUM_EPOCHS = 150          # epochs for the FINAL model (after HPO)
BATCH_SIZE = 32           # default / fallback batch size
CKPT_DIR   = "checkpoints"
SEED       = 42
print("Using device:", DEVICE)

# Resolve dataset root from the Roboflow download
try:
    DATASET_ROOT = dataset.location
except NameError:
    DATASET_ROOT = "/content/erotica-detection-4"

# --- HPO controls (RandomSearch) ---
MAX_TRIALS             = 10   # number of Optuna trials
SEARCH_EPOCHS          = 3    # short epochs per trial
SEARCH_MAX_BATCHES     = 150  # cap train batches/epoch during search (speed)
SEARCH_MAX_VAL_BATCHES = 50   # cap val batches when scoring a trial


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def collate_fn(batch):
    # Keep every image, including negatives with no boxes. torchvision's SSD
    # assigns all anchors to background for empty targets, and these "safe"
    # images are the hard negatives that suppress false positives.
    return tuple(zip(*batch))


def prepare_datasets():
    train_ds = RoboflowVOCDataset(root=DATASET_ROOT, split="train",
                                  transforms=SSDTransform(split="train"))
    val_ds = RoboflowVOCDataset(root=DATASET_ROOT, split="valid",
                                transforms=SSDTransform(split="valid"))
    print(f"Loaded {len(train_ds)} train and {len(val_ds)} val images.")
    return train_ds, val_ds


def build_loaders(train_ds, val_ds, batch_size):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, drop_last=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=collate_fn,
                            num_workers=2, pin_memory=True)
    return train_loader, val_loader


def run_one_epoch(model, loader, optimizer, cls_weight, max_batches=None):
    model.train()
    total, n = 0.0, 0
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        if batch is None:
            continue
        images, targets = batch
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = loss_dict["classification"] * cls_weight + loss_dict["bbox_regression"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item(); n += 1
    return total / max(1, n)


@torch.no_grad()
def eval_val_loss(model, loader, cls_weight, max_batches=None):
    # torchvision SSD only returns losses in train() mode. GroupNorm has no
    # running statistics, so train() + no_grad() yields losses with no side
    # effects on the model.
    model.train()
    total, n = 0.0, 0
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        if batch is None:
            continue
        images, targets = batch
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = loss_dict["classification"] * cls_weight + loss_dict["bbox_regression"]
        total += loss.item(); n += 1
    return total / max(1, n)

In [ ]:
import optuna

# Datasets are built once in the main cell and shared across all trials.
TRAIN_DATASET = None
VAL_DATASET = None


def ssd_hp_space(trial):
    """Sample one hyperparameter configuration.

    Mirrors the tuning scale of the other _hyperparameters notebooks
    (lr, weight_decay, batch_size) plus SGD momentum and the detector-specific
    classification-loss weight (the 1.5 factor hard-coded in the base notebook).
    """
    return {
        "learning_rate": trial.suggest_float("learning_rate", 5e-4, 1e-2, log=True),
        "momentum":      trial.suggest_float("momentum", 0.80, 0.99, step=0.01),
        "weight_decay":  trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        "cls_weight":    trial.suggest_float("cls_weight", 1.0, 2.5, step=0.5),
        "batch_size":    trial.suggest_categorical("batch_size", [16, 32]),
    }


def objective(trial):
    hp = ssd_hp_space(trial)
    print(f"\n--- Trial {trial.number} --- {hp}")
    set_seed(SEED + trial.number)

    train_loader, val_loader = build_loaders(TRAIN_DATASET, VAL_DATASET, hp["batch_size"])
    model = build_model(NUM_CLASSES, IMG_SIZE).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=hp["learning_rate"],
                                momentum=hp["momentum"], weight_decay=hp["weight_decay"])

    best_map = -1.0
    try:
        for epoch in range(SEARCH_EPOCHS):
            tr = run_one_epoch(model, train_loader, optimizer, hp["cls_weight"], SEARCH_MAX_BATCHES)
            # Score trials by validation mAP@50-95 (the reported metric), not
            # val loss, which tracks mAP poorly for detection.
            metrics = evaluate_map(model, val_loader, max_batches=SEARCH_MAX_VAL_BATCHES)
            vmap = metrics["map"].item()
            best_map = max(best_map, vmap)
            print(f"  trial {trial.number} epoch {epoch+1}/{SEARCH_EPOCHS}: "
                  f"train_loss {tr:.4f} | val_mAP {vmap:.4f}")

            # Pruning: stop unpromising trials early.
            trial.report(vmap, epoch)
            if trial.should_prune():
                print(f"  trial {trial.number} pruned at epoch {epoch+1}")
                raise optuna.TrialPruned()
        return best_map
    finally:
        del model, optimizer, train_loader, val_loader
        _free_cuda()


def run_hyperparameter_search(max_trials=MAX_TRIALS):
    sampler = optuna.samplers.RandomSampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=max_trials)

    print("\n" + "=" * 60)
    print("BEST TRIAL")
    print("=" * 60)
    print(f"  val_mAP@50-95: {study.best_value:.4f}")
    print(f"  params:   {study.best_params}")
    return study

In [ ]:
import csv
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchmetrics.detection.mean_ap import MeanAveragePrecision


@torch.no_grad()
def evaluate_map(model, loader, max_batches=None):
    model.eval()
    metric = MeanAveragePrecision(max_detection_thresholds=[1, 10, 100]).cpu()
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        if batch is None:
            continue
        images, targets = batch
        images = [img.to(DEVICE) for img in images]
        outputs = model(images)
        outputs = [{k: v.cpu() for k, v in o.items()} for o in outputs]
        targets = [{k: v.cpu() for k, v in t.items()} for t in targets]
        metric.update(outputs, targets)
    return metric.compute()


def train_final_model(best_params, train_ds, val_ds):
    print("\n" + "=" * 60)
    print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
    print("=" * 60)
    set_seed(SEED)

    bs = best_params.get("batch_size", BATCH_SIZE)
    cls_weight = best_params.get("cls_weight", 1.5)
    train_loader, val_loader = build_loaders(train_ds, val_ds, bs)

    model = build_model(NUM_CLASSES, IMG_SIZE).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(),
                                lr=best_params.get("learning_rate", 2e-3),
                                momentum=best_params.get("momentum", 0.9),
                                weight_decay=best_params.get("weight_decay", 5e-4))
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    os.makedirs("results", exist_ok=True)
    os.makedirs(CKPT_DIR, exist_ok=True)
    csv_path = os.path.join("results", "metrics.csv")
    with open(csv_path, "w", newline="") as f:
        csv.writer(f).writerow(["Epoch", "Train Loss", "LR",
                                "Val mAP@50", "Val mAP@50-95", "Val mAP@75"])

    # Resume from the last checkpoint if present.
    start_epoch, best_map = 0, -1.0
    last_ckpt = os.path.join(CKPT_DIR, "last.pth")
    if os.path.exists(last_ckpt):
        ck = torch.load(last_ckpt, map_location=DEVICE)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        start_epoch = ck["epoch"]; best_map = ck.get("best_map", -1.0)
        print(f"Resumed from epoch {start_epoch + 1}/{NUM_EPOCHS}")

    for epoch in range(start_epoch, NUM_EPOCHS):
        t0 = time.time()
        tr = run_one_epoch(model, train_loader, optimizer, cls_weight)
        scheduler.step()
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Loss: {tr:.4f} | "
              f"LR: {lr:.6f} | {time.time()-t0:.1f}s")

        m50 = m5095 = m75 = ""
        if (epoch + 1) % 5 == 0:
            metrics = evaluate_map(model, val_loader)
            m50, m5095, m75 = (metrics["map_50"].item(),
                               metrics["map"].item(),
                               metrics["map_75"].item())
            print(f"  Val -> mAP@50: {m50:.4f} | mAP@50-95: {m5095:.4f} | mAP@75: {m75:.4f}")
            if m5095 > best_map:
                best_map = m5095
                torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best.pth"))
                print(f"  ✔ new best mAP@50-95: {best_map:.4f} (saved best.pth)")

        with open(csv_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch + 1, tr, lr, m50, m5095, m75])

        # Per-epoch checkpoint so a Colab disconnect doesn't lose the run.
        torch.save({"epoch": epoch + 1, "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "best_map": best_map, "classes": CLASSES},
                   last_ckpt)

    # Load best-by-mAP weights before returning.
    best_path = os.path.join(CKPT_DIR, "best.pth")
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        print(f"Loaded best weights (mAP@50-95: {best_map:.4f}).")
    return model

In [ ]:
# ============================================================
# Run the full pipeline: HPO search -> final training
# ============================================================
set_seed(SEED)
TRAIN_DATASET, VAL_DATASET = prepare_datasets()

study = run_hyperparameter_search(MAX_TRIALS)
best_params = study.best_params

model = train_final_model(best_params, TRAIN_DATASET, VAL_DATASET)

In [ ]:
import json

# ============================================================
# Test-set evaluation with the best model
# ============================================================
test_ds = RoboflowVOCDataset(root=DATASET_ROOT, split="test",
                             transforms=SSDTransform(split="test"))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)

test_metrics = evaluate_map(model, test_loader)
print("=== TEST RESULTS ===")
print(f"mAP@50:    {test_metrics['map_50'].item():.4f}")
print(f"mAP@50-95: {test_metrics['map'].item():.4f}")
print(f"mAP@75:    {test_metrics['map_75'].item():.4f}")

os.makedirs("results", exist_ok=True)
with open(os.path.join("results", "test_metrics.csv"), "w") as f:
    f.write("metric,value\n")
    f.write(f"map_50,{test_metrics['map_50'].item()}\n")
    f.write(f"map_50_95,{test_metrics['map'].item()}\n")
    f.write(f"map_75,{test_metrics['map_75'].item()}\n")

with open(os.path.join("results", "best_hyperparameters.json"), "w") as f:
    json.dump({"best_params": best_params, "best_val_map": study.best_value}, f, indent=4)
print("Saved test_metrics.csv and best_hyperparameters.json")

In [ ]:
import json

# ============================================================
# Save deployment artifacts (weights + config)
# ============================================================
output_dir = "conversion"
os.makedirs(output_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(output_dir, "ssd_mbv3_large.pth"))
torch.save({"epoch": NUM_EPOCHS, "model": model.state_dict(),
            "classes": CLASSES, "hyperparameters": best_params},
           os.path.join(output_dir, "ssd_mbv3_large_full.pth"))

with open(os.path.join(output_dir, "labels.json"), "w") as f:
    json.dump(CLASSES, f, indent=4)

train_config = {
    "model_architecture": "ssdlite320_mobilenet_v3_large",
    "dataset": "Roboflow VOC",
    "num_classes": NUM_CLASSES,
    "class_names": CLASSES,
    "input_size": IMG_SIZE,
    "epochs": NUM_EPOCHS,
    "optimizer": "SGD",
    "scheduler": "CosineAnnealingLR",
    "best_hyperparameters": best_params,
    "device": str(DEVICE),
}
with open(os.path.join(output_dir, "training_config.json"), "w") as f:
    json.dump(train_config, f, indent=4)

print(f"Saved deployment artifacts to {output_dir}/")

In [ ]:
# Zip results + conversion artifacts and download
!zip -r ssdmobilenet_hpo_results.zip results/ conversion/

from google.colab import files
files.download('ssdmobilenet_hpo_results.zip')

In [ ]:
!pip install onnx onnxruntime onnxscript

In [ ]:
import torch

# ONNX export must run with the model AND the dummy input on the SAME device,
# and torchvision's SSD post-processing only traces with the legacy exporter
# (dynamo=False); the dynamo exporter errors on it.
model.eval()
model.cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

torch.onnx.export(
    model,
    dummy,
    "ssd_mbv3_hpo.onnx",
    input_names=["input"],
    output_names=["boxes", "scores", "labels"],
    opset_version=18,
    dynamo=False,
)
print("Exported ssd_mbv3_hpo.onnx")
model.to(DEVICE)

from google.colab import files
files.download("ssd_mbv3_hpo.onnx")